# Acceptance: `sum_by`, one name for summing across strata

**This is a sign-off page, not a tutorial.** It checks the rename of summer4's
"add up the strata" operations (plan `plans/sum-by.plan.md`).

Two names used to read wrongly. `total()` sounded like a single number but kept
time; `sum_over(age)` sounded like it removed age but kept it. Both are now
`sum_by`, named for what it **keeps**, as in pandas' `groupby(...).sum()`.
The old spellings were removed, so the table below is the migration map:

| Old | New | Keeps |
|---|---|---|
| `.total()` | `.sum_by()` | time |
| `.sum_over(age)` | `.sum_by(age)` | time, age |
| — | `.sum_by(age, strain)` | time, age, strain |
| `FlowRef(f).sum()` / `.sum_over(age)` | `FlowRef(f).sum_by()` / `.sum_by(age)` | (rate tree) |
| `Compartments(sum_over=age)`, `FlowMass(..., sum_over=...)`, `Reduce(sum_over=age)` | `group_by=` | |

Time is never summed by `sum_by`. Reducing over time is a separate step:
`integrate()`, `cumulative_integral()`, `rolling()`, `at()`.

Nothing here asserts; the checks are in `tests/test_sum_by.py`. Run every cell top
to bottom (a few seconds). Each section ends with a **What to check** box.

| | You are checking that… |
|---|---|
| A | `sum_by()` keeps time, `sum_by(age)` keeps age, and `sum_by(age, strain)` keeps both, and they add up |
| B | a single number over time comes from a separate time reduction |

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import FlowModel, FlowRef, Param, Property, PropertyMap, Split, TransitionFlow
from summer4.epi import ForceOfInfection

state = Property("state", ("S", "I", "R"))
age = Property("age", ("child", "adult"))
strain = Property("strain", ("a", "b"))
# Only infectious compartments carry a strain, so the map is ragged.
pmap = PropertyMap.from_property(state).stratify(age).stratify(strain, where=state["I"])

model = FlowModel(pmap)
for s in strain.traits:
    model.add_flow(
        TransitionFlow(
            f"infection_{s}",
            state["S"],
            state["I"] & strain[s],
            ForceOfInfection(f"foi_{s}", infectious=state["I"] & strain[s], contact_rate=Param("beta")),
        )
    )
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
model.set_initial_population(
    {age["child"]: 1000.0, age["adult"]: 1500.0},
    [Split(state, {"S": 0.98, "I": 0.02, "R": 0.0}), Split(strain, {"a": 0.7, "b": 0.3}, where=state["I"])],
)
res = model.compile().run({"beta": 0.25}, t0=0.0, t1=120.0, dt=1.0)
pmap.labels()

## A. What `sum_by` keeps

Infectious people, three ways. The first plot is `sum_by(age, strain)`: one line
per (age, strain) pair. The second puts `sum_by(age)` and `sum_by()` on top of
the sums of those lines, which they should equal exactly. The bar chart shows how
many columns each spelling keeps.

In [ ]:
infectious = res["compartments"].select(state["I"])
by_both = infectious.sum_by(age, strain)
by_age = infectious.sum_by(age)
everyone = infectious.sum_by()

days = np.asarray(infectious.times.values)
both = pd.DataFrame(np.asarray(by_both.values.data), index=pd.Index(days, name="day"), columns=by_both.values.pmap.labels())
both.plot(title="A. Infectious people: sum_by(age, strain)", labels={"value": "people", "variable": ""}).show()

check = pd.DataFrame(
    {
        "sum_by(age): child": np.asarray(by_age.values.data)[:, 0],
        "child columns of sum_by(age, strain), added": both.filter(like="age=child").sum(axis=1).to_numpy(),
        "sum_by()": np.asarray(everyone.values),
        "all columns of sum_by(age, strain), added": both.sum(axis=1).to_numpy(),
    },
    index=pd.Index(days, name="day"),
)
fig = check.plot(title="A. sum_by(age) and sum_by() equal the added-up columns", labels={"value": "people", "variable": ""})
for trace in (fig.data[1], fig.data[3]):
    trace.update(line={"dash": "dash", "width": 3})
fig.show()

pd.Series(
    {"sum_by()": 1, "sum_by(age)": by_age.values.data.shape[-1], "sum_by(age, strain)": by_both.values.data.shape[-1]},
    name="columns kept besides time",
).plot.bar(title="A. Columns kept besides time", labels={"index": "spelling", "value": "columns"}).show()
pd.DataFrame(
    {"dims": [str(everyone.dims), str(by_age.dims), str(by_both.dims)],
     "labels": ["—", by_age.values.pmap.labels(), by_both.values.pmap.labels()]},
    index=["sum_by()", "sum_by(age)", "sum_by(age, strain)"],
)

> **What to check (A).** The first plot has four lines (two ages × two
> strains). In the second, each dashed line lies on the solid line next to it in
> the legend. The bars are 1, 2 and 4, and the dims all start with `time`.

## B. A single number needs a time reduction

`sum_by()` on the new-infection flows gives the infection rate per day, all ages
and strains together — a time series. Total infections over the run come from
integrating that over time. In this model that total is also how far the
susceptibles fell, which the table compares. `cumulative_integral()` is the
running total: the same integral up to each day.

In [ ]:
rate = res["infection_a"].sum_by() + res["infection_b"].sum_by()
total = float(np.asarray(rate.integrate().values))
susceptible = np.asarray(res["compartments"].select(state["S"]).sum_by().values)

pd.DataFrame(
    {"infections per day (sum_by())": np.asarray(rate.values), "running total (cumulative_integral())": np.asarray(rate.cumulative_integral().values)},
    index=pd.Index(days, name="day"),
).plot(title="B. Infection rate, and its running total", labels={"value": "people (per day, or total)", "variable": ""}).show()
pd.Series(
    {
        "integrate(): total infections": round(total, 1),
        "susceptibles at start - at end": round(float(susceptible[0] - susceptible[-1]), 1),
        "dims after integrate()": rate.integrate().dims,
    },
    name="B",
    dtype=object,
)

> **What to check (B).** The rate rises and falls; the running total rises to
> a plateau at the `integrate()` total. `integrate()` has dims `()` (one number) and is within about 1% of
> the fall in susceptibles (the trapezoid rule over daily saves, not the solver's
> own sum).